In [1]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import torchvision
import torchvision.transforms as transforms
from torch.utils.data import DataLoader, Subset
import numpy as np
import copy
import time
import random
import os
import sys
import warnings

# Suppress annoying warnings
warnings.filterwarnings("ignore")

# ==========================================
# 1. KFN ARCHITECTURE (USED FOR EQUAL CAPACITY BASELINE)
# ==========================================
# We use the exact same expanded model so the parameter count (~5.6M) 
# is mathematically identical to your KFN method for a fair baseline.

class CosineLinear(nn.Module):
    def __init__(self, in_features, out_features, sigma=10.0):
        super().__init__()
        self.weight = nn.Parameter(torch.Tensor(out_features, in_features))
        self.sigma = nn.Parameter(torch.Tensor([sigma]))
        nn.init.kaiming_uniform_(self.weight, a=np.sqrt(5))
    def forward(self, input):
        return self.sigma * F.linear(F.normalize(input, p=2, dim=1),
                                     F.normalize(self.weight, p=2, dim=1))

class ResidualBlock(nn.Module):
    def __init__(self, in_c, out_c, stride=1):
        super().__init__()
        self.conv1 = nn.Conv2d(in_c, out_c, 3, stride, 1, bias=False)
        self.bn1 = nn.BatchNorm2d(out_c)
        self.conv2 = nn.Conv2d(out_c, out_c, 3, 1, 1, bias=False)
        self.bn2 = nn.BatchNorm2d(out_c)
        self.shortcut = nn.Sequential()
        if stride != 1 or in_c != out_c:
            self.shortcut = nn.Sequential(nn.Conv2d(in_c, out_c, 1, stride, bias=False),
                                          nn.BatchNorm2d(out_c))
    def forward(self, x): return F.relu(self.bn1(self.conv1(x)) + self.shortcut(x))

class Specialist(nn.Module):
    def __init__(self):
        super().__init__()
        self.conv1 = nn.Conv2d(3, 64, 3, 1, 1, bias=False)
        self.bn1 = nn.BatchNorm2d(64)
        self.relu = nn.ReLU()
        self.layer1 = self._make_layer(64, 64, 2)
        self.layer2 = self._make_layer(64, 128, 2, stride=2)
        self.layer3 = self._make_layer(128, 256, 2, stride=2)
        self.pool = nn.AdaptiveAvgPool2d((4, 4))
        self.final_conv = nn.Conv2d(256, 64, 1)
    def _make_layer(self, in_c, out_c, blocks, stride=1):
        layers = [ResidualBlock(in_c, out_c, stride)]
        for _ in range(1, blocks): layers.append(ResidualBlock(out_c, out_c))
        return nn.Sequential(*layers)
    def forward(self, x):
        return self.final_conv(self.pool(self.layer3(self.layer2(self.layer1(self.relu(self.bn1(self.conv1(x))))))))

class StructuralFusionModule(nn.Module):
    def __init__(self, in_channels, old_dim, new_dim, novelty_score):
        super().__init__()
        self.novelty_score = novelty_score
        self.has_expansion = new_dim > 0
        self.proj_reuse = nn.Sequential(nn.Conv2d(in_channels, old_dim, 1, bias=False),
                                        nn.BatchNorm2d(old_dim), nn.ReLU(), nn.Dropout(0.1))
        self.gate_reuse = nn.Parameter(torch.tensor([0.0]))
        if self.has_expansion:
            self.proj_expand = nn.Sequential(nn.Conv2d(in_channels, new_dim, 1, bias=False),
                                             nn.BatchNorm2d(new_dim), nn.ReLU())
    def forward(self, x, old_memory_detached):
        reuse_scale = max(0.1, 1.0 - self.novelty_score)
        delta_old = self.proj_reuse(x) * torch.sigmoid(self.gate_reuse) * reuse_scale
        feat_new = self.proj_expand(x) - self.proj_expand(x).mean(dim=(2,3), keepdim=True) if self.has_expansion else None
        return delta_old, feat_new

class GlobalModel(nn.Module):
    def __init__(self, n_specs, ch, n_classes, old_dim=64, new_dim=0, novelty_score=0.0):
        super().__init__()
        self.old_proj = nn.ModuleList([nn.Conv2d(ch, ch, 1) for _ in range(n_specs)])
        self.old_gate = nn.Sequential(nn.AdaptiveAvgPool2d(1),
                                      nn.Conv2d(ch, ch//4, 1),
                                      nn.ReLU(),
                                      nn.Conv2d(ch//4, ch, 1),
                                      nn.Sigmoid())
        self.old_bottleneck = nn.Sequential(nn.Conv2d(ch, old_dim, 1), nn.ReLU())
        self.old_dim = old_dim
        self.new_dim = new_dim
        self.fusion = StructuralFusionModule(ch, old_dim, new_dim, novelty_score) if new_dim > 0 else None
        self.pool = nn.AdaptiveAvgPool2d(1)
        self.classifier = CosineLinear(old_dim + new_dim, n_classes)
    def forward_old(self, feats):
        p = self.old_proj[0](feats[0])
        return self.old_bottleneck(p * self.old_gate(p))
    def forward(self, feats):
        old_mem = self.forward_old(feats)
        if self.fusion:
            delta, f_new = self.fusion(feats[-1], old_mem.detach())
            final = torch.cat([old_mem + delta, f_new], dim=1) if f_new is not None else old_mem + delta
        else:
            final = old_mem
        W_old = self.classifier.weight[:5, :self.old_dim]
        W_new = self.classifier.weight[5:, :]
        logits_old = F.linear(F.normalize(self.pool(old_mem).flatten(1), p=2, dim=1),
                              F.normalize(W_old, p=2, dim=1)) * self.classifier.sigma
        logits_new = F.linear(F.normalize(self.pool(final).flatten(1), p=2, dim=1),
                              F.normalize(W_new, p=2, dim=1)) * self.classifier.sigma
        return torch.cat([logits_old, logits_new], dim=1), old_mem

class BaselineNet(nn.Module):
    # Initializes full capacity up front for standard baseline training
    def __init__(self, n_classes=10, n_specs=2, old_dim=64, new_dim=128, novelty_score=0.99):
        super().__init__()
        self.specialists = nn.ModuleList([Specialist() for _ in range(n_specs)])
        self.global_model = GlobalModel(n_specs, 64, n_classes, old_dim, new_dim, novelty_score)
    def forward(self, x):
        return self.global_model([s(x) for s in self.specialists])

# ==========================================
# 2. UTILITIES
# ==========================================
def set_deterministic(seed):
    random.seed(seed); np.random.seed(seed); torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)
        torch.backends.cudnn.deterministic = True
        torch.backends.cudnn.benchmark = False

def evaluate(model, loader, device):
    model.eval(); correct = 0; total = 0
    with torch.no_grad():
        for x, y in loader:
            x, y = x.to(device), y.to(device)
            if getattr(device, "type", "") == 'cuda':
                with torch.amp.autocast('cuda'):
                    logits, _ = model(x)
            else:
                logits, _ = model(x)
            correct += (logits.argmax(1) == y).sum().item()
            total += y.size(0)
    return 100*correct/total

def get_kaggle_dataset_robust(transform_train, transform_test):
    try:
        data_root = '/kaggle/input/cifar-10'
        train_ds = torchvision.datasets.CIFAR10(root=data_root, train=True, download=False, transform=transform_train)
        test_ds = torchvision.datasets.CIFAR10(root=data_root, train=False, download=False, transform=transform_test)
        print(f"✅ Offline Data Loaded successfully from: {data_root}")
        return train_ds, test_ds
    except Exception as e:
        print(f"\n❌ CRITICAL ERROR: Could not load dataset. Details: {e}")
        sys.exit(1)

# ==========================================
# 3. EXPERIENCE REPLAY BASELINE EXPERIMENT
# ==========================================
def run_er_baseline(seed, loaders, epochs_A, epochs_B, device):
    set_deterministic(seed)
    l_A, l_B, t_A, t_B, replay_loader = loaders
    
    scaler = torch.amp.GradScaler('cuda') if getattr(device, "type", "") == 'cuda' else None

    # Instantiate Model with full capacity exactly matching expanded KFN (new_dim=128)
    model = BaselineNet(n_classes=10, n_specs=2, new_dim=128).to(device)
    
    # --- TASK A TRAINING ---
    opt = torch.optim.AdamW(model.parameters(), lr=1e-3, weight_decay=1e-4)
    
    for ep in range(epochs_A):
        model.train()
        for x, y in l_A:
            x, y = x.to(device), y.to(device)
            opt.zero_grad()
            if getattr(device, "type", "") == 'cuda':
                with torch.amp.autocast('cuda'):
                    out, _ = model(x)
                    loss = F.cross_entropy(out, y)
                scaler.scale(loss).backward()
                scaler.step(opt)
                scaler.update()
            else:
                out, _ = model(x)
                loss = F.cross_entropy(out, y)
                loss.backward()
                opt.step()

    acc_A_init = evaluate(model, t_A, device)

    # --- TASK B TRAINING (WITH REPLAY) ---
    rep_iter = iter(replay_loader)
    
    for ep in range(epochs_B):
        model.train()
        for x_B, y_B in l_B:
            x_B, y_B = x_B.to(device), y_B.to(device)
            
            # Fetch Replay Batch
            try:
                x_A, y_A = next(rep_iter)
            except StopIteration:
                rep_iter = iter(replay_loader)
                x_A, y_A = next(rep_iter)
            x_A, y_A = x_A.to(device), y_A.to(device)

            opt.zero_grad()
            
            # We process Task A and Task B sequentially to prevent VRAM OOM from 512 batch size
            if getattr(device, "type", "") == 'cuda':
                # Replay Loss (Task A)
                with torch.amp.autocast('cuda'):
                    out_A, _ = model(x_A)
                    loss_A = F.cross_entropy(out_A, y_A)
                scaler.scale(loss_A).backward()
                
                # Current Task Loss (Task B)
                with torch.amp.autocast('cuda'):
                    out_B, _ = model(x_B)
                    loss_B = F.cross_entropy(out_B, y_B)
                scaler.scale(loss_B).backward()
                
                scaler.step(opt)
                scaler.update()
            else:
                out_A, _ = model(x_A)
                loss_A = F.cross_entropy(out_A, y_A)
                loss_A.backward()
                
                out_B, _ = model(x_B)
                loss_B = F.cross_entropy(out_B, y_B)
                loss_B.backward()
                
                opt.step()

    # Final Evaluation
    acc_A_final = evaluate(model, t_A, device)
    acc_B_final = evaluate(model, t_B, device)

    return {
        "acc_A_init": acc_A_init, 
        "acc_A_final": acc_A_final, 
        "acc_B_final": acc_B_final
    }

# ==========================================
# 4. EXECUTION & REPORTING
# ==========================================
def run_all():
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    print(f"🚀 Environment Configured: {device}")

    # Identical Augmentations to KFN
    stats = ((0.4914, 0.4822, 0.4465), (0.247, 0.243, 0.261))
    t_train = transforms.Compose([
        transforms.RandomCrop(32, padding=4),
        transforms.RandomHorizontalFlip(),
        transforms.ColorJitter(brightness=0.2, contrast=0.2, saturation=0.2), 
        transforms.ToTensor(),
        transforms.Normalize(*stats)
    ])
    t_test = transforms.Compose([transforms.ToTensor(), transforms.Normalize(*stats)])

    train_ds, test_ds = get_kaggle_dataset_robust(t_train, t_test)

    # Setup matches KFN Phase 1 (10) and Phase 3 (40)
    EPOCHS_A = 10
    EPOCHS_B = 40
    REPLAY_SIZE = 12000
    BS = 256
    NW = 2 # Multiprocessing enabled to prevent CPU bottleneck

    loaders = (
        DataLoader(Subset(train_ds, [i for i,l in enumerate(train_ds.targets) if l<5]), BS, shuffle=True, num_workers=NW, pin_memory=True, persistent_workers=True),
        DataLoader(Subset(train_ds, [i for i,l in enumerate(train_ds.targets) if l>=5]), BS, shuffle=True, num_workers=NW, pin_memory=True, persistent_workers=True),
        DataLoader(Subset(test_ds, [i for i,l in enumerate(test_ds.targets) if l<5]), BS, num_workers=NW, pin_memory=True),
        DataLoader(Subset(test_ds, [i for i,l in enumerate(test_ds.targets) if l>=5]), BS, num_workers=NW, pin_memory=True),
        DataLoader(Subset(train_ds, [i for i,l in enumerate(train_ds.targets) if l<5][:REPLAY_SIZE]), BS, shuffle=True, num_workers=NW, pin_memory=True, persistent_workers=True)
    )

    print("\n[1] Running Multi-Seed ER Baseline Experiments...")
    seeds = [0, 1, 2, 3, 4]
    results = []
    
    for s in seeds:
        print(f" -> Training ER Baseline Seed {s}...")
        res = run_er_baseline(s, loaders, EPOCHS_A, EPOCHS_B, device)
        results.append(res)
        print(f"    ↳ Seed {s} Results: Task A Init = {res['acc_A_init']:.2f}% | Task A Final = {res['acc_A_final']:.2f}% | Task B Final = {res['acc_B_final']:.2f}%")

    # ==========================================
    # FINAL PAPER-READY REPORTING
    # ==========================================
    print("\n" + "="*80)
    print("SECTION A: Experience Replay Baseline (Mean ± Std over 5 Seeds)")
    print("="*80)
    
    for k, label in [("acc_A_init", "Task A Init"), ("acc_A_final", "Task A Final"), ("acc_B_final", "Task B Final")]:
        vals = [r[k] for r in results]
        print(f"{label:15}: {np.mean(vals):.2f}% ± {np.std(vals):.2f}")

    ret_vals = [(r['acc_A_final']/r['acc_A_init'])*100 for r in results]
    forg_vals = [r['acc_A_init'] - r['acc_A_final'] for r in results]
    bwt_vals = [r['acc_A_final'] - r['acc_A_init'] for r in results]

    print(f"Retention:      {np.mean(ret_vals):.2f}% ± {np.std(ret_vals):.2f}")
    print(f"Forgetting:     {np.mean(forg_vals):.2f} ± {np.std(forg_vals):.2f}")
    print(f"Backward Trans: {np.mean(bwt_vals):.2f} ± {np.std(bwt_vals):.2f}")
    print("="*80)

if __name__ == "__main__":
    run_all()

🚀 Environment Configured: cuda
✅ Offline Data Loaded successfully from: /kaggle/input/cifar-10

[1] Running Multi-Seed ER Baseline Experiments...
 -> Training ER Baseline Seed 0...
    ↳ Seed 0 Results: Task A Init = 78.56% | Task A Final = 82.28% | Task B Final = 55.70%
 -> Training ER Baseline Seed 1...
    ↳ Seed 1 Results: Task A Init = 77.30% | Task A Final = 79.38% | Task B Final = 74.82%
 -> Training ER Baseline Seed 2...
    ↳ Seed 2 Results: Task A Init = 83.60% | Task A Final = 84.40% | Task B Final = 52.94%
 -> Training ER Baseline Seed 3...
    ↳ Seed 3 Results: Task A Init = 81.26% | Task A Final = 82.58% | Task B Final = 65.38%
 -> Training ER Baseline Seed 4...
    ↳ Seed 4 Results: Task A Init = 84.26% | Task A Final = 75.46% | Task B Final = 74.36%

SECTION A: Experience Replay Baseline (Mean ± Std over 5 Seeds)
Task A Init    : 81.00% ± 2.72
Task A Final   : 80.82% ± 3.13
Task B Final   : 64.64% ± 9.12
Retention:      99.91% ± 5.33
Forgetting:     0.18 ± 4.42
Backward